## libraries ##

In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import researchpy as rp
from sklearn.preprocessing import RobustScaler

In [ ]:
file_path = '../data/creditcard.csv'
df=pd.read_csv(file_path)

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df['Amount'].describe()

In [ ]:
df['Time'].describe()

In [ ]:
df['Class'].value_counts()

In [ ]:
df.info()

### Data Visualation 

In [ ]:
#corelation analysis
plt.figure(figsize=(10,8))
corr = df.corr()['Class'].drop('Class')
corr.sort_values().plot(kind='barh')
plt.title('Corelation Analysis Class With Others')
plt.show()




In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x='Class', y='V17', data=df)
plt.title('V14 Değişkeninin Normal ve Fraud İşlemlere Göre Dağılımı 📦')
plt.show()

In [ ]:
columns = ['V14','V17','V12','V10','Amount','Time']
g = sns.PairGrid(df[columns])
g.map_diag(sns.histplot)
g.map_offdiag(sns.scatterplot)

In [ ]:


plt.figure(figsize=(10, 5))

# hue='Class' ile Normal ve Fraud vakalarını renklerine göre ayırıyoruz
sns.kdeplot(
    data=df,
    x='V14',
    hue='Class',
    fill=True,  # Eğrilerin altını renklendirir
    common_norm=False,  # Her iki sınıfın yoğunluğunu kendi içinde %100'e ölçekler
    palette={0: 'blue', 1: 'red'},  # 0: Mavi (Normal), 1: Kırmızı (Fraud)
)

plt.title('V14 Değişkeninin Sınıflara Göre Yoğunluk Dağılımı (KDE) 🌊')
plt.xlabel('V14 Değeri')
plt.ylabel('Yoğunluk (Density)')
plt.show()

### Statictical Analysis

In [ ]:
df.groupby('Class')[['Amount', 'Time']].agg(['mean','std','median','count'])

In [ ]:
df[['Time','Amount']].cov()

In [ ]:
import statsmodels.stats.api as sms
df['Class'].mean()

In [ ]:
sms.DescrStatsW(df['Class']).tconfint_mean()

In [ ]:
#hypotesis test 
#h0 data is normall h1 data is not normall
from scipy.stats import shapiro
shapiro(df)

In [ ]:
from scipy import stats

def calc_effect_size(df,features):
    group_0 = df[df['Class']==0][features]
    group_1 = df[df['Class']==1][features]
    u_stat, p_val = stats.mannwhitneyu(group_0,group_1)
    n1,n2 = len(group_0),len(group_1)
    mean_u = n1*n2/2
    std_u = np.sqrt((n1 * n2 * (n1 + n2 + 1)) / 12)
    z = (u_stat - mean_u) / std_u

    # r = Z / sqrt(N)
    r = z / np.sqrt(n1 + n2)

    return abs(r), p_val

In [ ]:
columns=df.columns
col=[]
for name in columns:
    if name != 'Class':
        col.append(name)
    else:
        continue    


In [ ]:
for i in col:
    r_val,p_val = calc_effect_size(df,i)
    print(f"{i} degiskeni için r value degeri: {r_val} | p value degeri {p_val}")


## 📊 İstatistiksel Analiz ve Öznitelik İncelemesi Raporu

### 1. Normallik Testi (Shapiro-Wilk) 🧪
- **Hipotezler:**
  - $H_0$: Değişken normal dağılıma sahiptir.
  - $H_1$: Değişken normal dağılıma sahip değildir.
- **Sonuç:** Yapılan Shapiro-Wilk testi sonucunda $p < 0.05$ elde edilmiş ve $H_0$ hipotezi reddedilmiştir. Verilerin normal dağılmadığı tespit edildiği için analizlerde **parametrik olmayan (non-parametric)** yöntemler tercih edilmiştir.

---

### 2. İki Grup Karşılaştırması ve Etki Büyüklüğü ($r$) 📐
Normal (`Class=0`) ve Dolandırıcılık (`Class=1`) işlemleri arasındaki farkı ölçmek için **Mann-Whitney U** testi uygulanmış ve etki büyüklüğü ($r$) hesaplanmıştır.

- **Hipotezler:**
  - $H_0$: Normal ve Fraud grupları arasında değişken açısından fark yoktur ($M_0 = M_1$).
  - $H_1$: İki grup arasında istatistiksel olarak anlamlı bir fark vardır ($M_0 \neq M_1$).
- **Etki Büyüklüğü Formülü:** 
  $$r = \frac{\vert{}Z\vert{}}{\sqrt{N}}$$
  *(Devasa örneklem boyutunun ($N > 284.000$) yarattığı illüzyonu ortadan kaldırarak gerçek ayırt etme gücünü verir.)*

---

### 3. Öznitelik Önem Düzeyleri ve Bulgular 🏆

| Öznitelik Tipi | Öne Çıkan Değişkenler | İstatistiksel Değerlendirme |
| :--- | :--- | :--- |
| **En Güçlü Ayrıştıranlar** | `V14` ($r = 0.0646$), `V4` ($r = 0.0630$), `V12` ($r = 0.0628$), `V11` ($r = 0.0601$) | $p < 0.05$ ve en yüksek $r$ değerleri. Grupları en belirgin ayıran özniteliklerdir. |
| **İstatistiki Olarak Anlamsızlar** | `V15` ($p \approx 0.129$), `V22` ($p \approx 0.266$), `V13` ($p \approx 0.051$) | $p \ge 0.05$ olduğu için $H_0$ reddedilemez. Gruplar arasında anlamlı fark yoktur. |

In [ ]:
Q1 = df['V14'].quantile(0.25)
Q3 = df['V14'].quantile(0.75)
IOR = Q3 - Q1
print(f"Q1: {Q1}  Q2: {Q3}  IOR:{IOR}")


In [ ]:
alt_sinir = Q1 - 1.5*IOR
ust_sinir = Q3 + 1.5*IOR
print(f"Alt sinir: {alt_sinir}  Ust sinir: {ust_sinir}")

In [ ]:
outlier_values = df[(df['V14']< alt_sinir) | (df['V14']>ust_sinir)]
print(len(outlier_values))
outlier_values['Class'].value_counts()

Aykırı değer analizinde elde edilen bulgulara bakıldığında aykırı olarak bulunan fraud işlemler tüm fraud işlemlerin çok büyük bir kısmını oluşturmaktadır bu sebeple aykırı gözlem analizini daha çok normal olan işlemler için aykırı değer eşiğini yukarı çekerek tekrar analiz etmeliyiz 


In [ ]:
alt_sinir = Q1 - 3*IOR
ust_sinir = Q3 + 3*IOR
print(f"guncel alt sinir: {alt_sinir}, guncel ust sinir: {ust_sinir}")
outlier_values = df[(df['V14']<alt_sinir)| (df['V14']>ust_sinir)]
outlier_values['Class'].value_counts()

aykırı olarak değerlendirdigimiz verileri silmeme kararı alındı 

In [ ]:
robust_scaler = RobustScaler(quantile_range=(25.0,75.0), with_centering=True,with_scaling=True)
df['scaled_Amount'] = robust_scaler.fit_transform(df[['Amount']])
df['scaled_Time'] = robust_scaler.fit_transform(df[['Time']])
df.drop(['Amount','Time'],axis=1,inplace=True)